In [204]:
import os

print(os.getcwd())

/home/naothoi/Downloads/LES_smagorinsky


In [205]:
os.listdir()

['computational_grid_gmsh_visualized.py',
 '.ipynb_checkpoints',
 'solver_finite_volume.ipynb',
 'Untitled1.ipynb',
 'processed_mesh.npz',
 'PISO_SOLVER.ipynb',
 'Untitled2.ipynb',
 'Untitled.ipynb',
 'untitled',
 'computational_grid_gmsh_visualized.ipynb',
 'fluid_mesh_3d.msh',
 'read_mesh_from_the_generated_mesh.ipynb']

In [206]:
import meshio

gmsh_mesh = meshio.read("fluid_mesh_3d.msh")

In [207]:
print(type(gmsh_mesh))

<class 'meshio._mesh.Mesh'>


In [208]:
print(gmsh_mesh.cells)

[<meshio CellBlock, type: triangle, num cells: 3068, tags: []>, <meshio CellBlock, type: triangle, num cells: 66, tags: []>, <meshio CellBlock, type: triangle, num cells: 132, tags: []>, <meshio CellBlock, type: triangle, num cells: 132, tags: []>, <meshio CellBlock, type: triangle, num cells: 130, tags: []>, <meshio CellBlock, type: triangle, num cells: 132, tags: []>, <meshio CellBlock, type: triangle, num cells: 66, tags: []>, <meshio CellBlock, type: tetra, num cells: 44172, tags: []>]


In [209]:
import numpy as np
import matplotlib.pyplot as plt

In [210]:
import meshio

mesh = meshio.read("fluid_mesh_3d.msh")

In [211]:
mesh = np.load("processed_mesh.npz")

points = mesh["points"]
tetra = mesh["tetra"]

cell_centroids = mesh["cell_centroids"]
cell_volumes = mesh["cell_volumes"]

unique_faces = mesh["unique_faces"]

owner = mesh["owner"]
neighbour = mesh["neighbour"]

face_centroids = mesh["face_centroids"]

face_area_vectors = mesh["face_area_vectors"]
face_areas = mesh["face_areas"]
face_normals = mesh["face_normals"]

print("Mesh loaded successfully.")

Mesh loaded successfully.


In [212]:
def initialize_fields(n_cells,
                      inlet_velocity=(1.0, 0.0, 0.0),
                      initial_pressure=0.0):
    """
    Initialize velocity and pressure fields.
    """

    velocity = np.zeros((n_cells, 3), dtype=np.float64)

    velocity[:, 0] = inlet_velocity[0]
    velocity[:, 1] = inlet_velocity[1]
    velocity[:, 2] = inlet_velocity[2]

    pressure = np.full(n_cells,
                       initial_pressure,
                       dtype=np.float64)

    return velocity, pressure

In [213]:
n_cells = len(tetra)

velocity, pressure = initialize_fields(n_cells)

In [214]:
print("Velocity shape :", velocity.shape)
print("Pressure shape :", pressure.shape)

print()

print("First five velocity vectors")

print(velocity[:5])

print()

print("First five pressure values")

print(pressure[:5])

Velocity shape : (44172, 3)
Pressure shape : (44172,)

First five velocity vectors
[[1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]

First five pressure values
[0. 0. 0. 0. 0.]


In [215]:
def interpolate_to_faces_central(phi,
                         owner,
                         neighbour):
    """
    Interpolate a cell-centered scalar or vector field
    to the mesh faces using central differencing.

    Parameters
    ----------
    phi : ndarray
        Cell-centered field.
        Shape:
            (n_cells,)      scalar
            (n_cells,3)     vector

    owner : ndarray
    neighbour : ndarray

    Returns
    -------
    phi_face
    """

    n_faces = len(owner)

    if phi.ndim == 1:

        phi_face = np.zeros(n_faces)

    else:

        phi_face = np.zeros((n_faces, phi.shape[1]))

    for f in range(n_faces):

        P = owner[f]
        N = neighbour[f]

        if N != -1:

            phi_face[f] = 0.5 * (phi[P] + phi[N])

        else:

            # Boundary face
            phi_face[f] = phi[P]

    return phi_face

In [216]:
velocity_face = interpolate_to_faces(
    velocity,
    owner,
    neighbour
)

print(velocity_face.shape)

print()

print(velocity_face[:5])

(90207, 3)

[[1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]


In [217]:
def compute_mass_flux(velocity_face,
                      face_area_vectors,
                      rho=1.0):
    """
    Compute the mass flux through every face.

    Parameters
    ----------
    velocity_face : ndarray (n_faces,3)

    face_area_vectors : ndarray (n_faces,3)

    rho : float

    Returns
    -------
    mass_flux : ndarray (n_faces,)
    """

    mass_flux = rho * np.sum(
        velocity_face * face_area_vectors,
        axis=1
    )

    return mass_flux

In [218]:
mass_flux = compute_mass_flux(
    velocity_face,
    face_area_vectors
)

print(mass_flux.shape)

print()

print(mass_flux[:10])

(90207,)

[ 2.37322609e-04 -3.05365923e-04 -1.41439332e-03  1.48243663e-03
 -9.32888121e-04  1.32138200e-03  2.19603958e-04 -6.08097840e-04
  2.59046225e-01 -4.59047656e-01]


In [219]:
def continuity_residual(mass_flux,
                        owner,
                        neighbour,
                        n_cells):
    """
    Compute the continuity residual for each control volume.

    Returns
    -------
    residual : ndarray (n_cells,)
    """

    residual = np.zeros(n_cells)

    for f in range(len(mass_flux)):

        P = owner[f]
        N = neighbour[f]

        residual[P] += mass_flux[f]

        if N != -1:
            residual[N] -= mass_flux[f]

    return residual

In [220]:
residual = continuity_residual(
    mass_flux,
    owner,
    neighbour,
    n_cells
)

print("Maximum continuity residual")
print(np.max(np.abs(residual)))

Maximum continuity residual
3.3306690738754696e-16


In [221]:
def compute_gradient(phi_face,
                     owner,
                     neighbour,
                     face_area_vectors,
                     cell_volumes):
    """
    Green-Gauss gradient reconstruction.

    Parameters
    ----------
    phi_face : ndarray (n_faces,)
        Scalar value on faces.

    owner : ndarray

    neighbour : ndarray

    face_area_vectors : ndarray (n_faces,3)

    cell_volumes : ndarray (n_cells,)

    Returns
    -------
    gradient : ndarray (n_cells,3)
    """

    n_cells = len(cell_volumes)

    gradient = np.zeros((n_cells,3))

    for f in range(len(owner)):

        P = owner[f]
        N = neighbour[f]

        contribution = phi_face[f] * face_area_vectors[f]

        gradient[P] += contribution

        if N != -1:

            gradient[N] -= contribution

    gradient /= cell_volumes[:,None]

    return gradient

In [222]:
pressure_face = interpolate_to_faces(
    pressure,
    owner,
    neighbour
)

pressure_gradient = compute_gradient(
    pressure_face,
    owner,
    neighbour,
    face_area_vectors,
    cell_volumes
)

In [223]:
print(pressure_gradient.shape)

print()

print(pressure_gradient[:10])

(44172, 3)

[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


In [224]:
phi = cell_centroids[:, 0]

In [225]:
phi_face = interpolate_to_faces(
    phi,
    owner,
    neighbour
)

In [226]:
grad_phi = compute_gradient(
    phi_face,
    owner,
    neighbour,
    face_area_vectors,
    cell_volumes
)

In [227]:
print("Gradient shape:", grad_phi.shape)

print("\nFirst five gradients:")
print(grad_phi[:5])

Gradient shape: (44172, 3)

First five gradients:
[[ 0.76974073  0.03380537 -0.00910566]
 [ 0.90668255 -0.07978712  0.2888779 ]
 [ 1.00390199 -0.00204641 -0.30763626]
 [ 0.75111024 -0.45167376  0.05458182]
 [ 0.87450375 -0.29602672  0.06112766]]


In [228]:
exact = np.array([1.0,0.0,0.0])

error = grad_phi - exact

error_norm = np.linalg.norm(error,axis=1)

print("Mean error :", np.mean(error_norm))
print("Max error  :", np.max(error_norm))
print("Min error  :", np.min(error_norm))

Mean error : 0.5001829688050956
Max error  : 3.949509060866996
Min error  : 0.011697790446417503


In [229]:
print("Mean dphidx =", np.mean(grad_phi[:,0]))
print("Mean dphidy =", np.mean(grad_phi[:,1]))
print("Mean dphidz =", np.mean(grad_phi[:,2]))

Mean dphidx = 1.0225620909005402
Mean dphidy = -4.240687134180537e-06
Mean dphidz = -0.0006117460982470593


In [230]:
def interpolate_to_faces_weighted(phi,
                                  owner,
                                  neighbour,
                                  cell_centroids,
                                  face_centroids):
    """
    Distance-weighted interpolation from cell centres to faces.

    Parameters
    ----------
    phi : ndarray
        Cell-centred field.
        Shape:
            (n_cells,)      -> scalar
            (n_cells,3)     -> vector

    owner : ndarray
        Owner cell index for each face.

    neighbour : ndarray
        Neighbour cell index (-1 for boundary faces).

    cell_centroids : ndarray (n_cells,3)

    face_centroids : ndarray (n_faces,3)

    Returns
    -------
    phi_face : ndarray
        Interpolated face values.
    """

    n_faces = len(owner)

    # Scalar field
    if phi.ndim == 1:
        phi_face = np.zeros(n_faces)

    # Vector field
    else:
        phi_face = np.zeros((n_faces, phi.shape[1]))

    for f in range(n_faces):

        P = owner[f]
        N = neighbour[f]

        # Boundary face
        if N == -1:

            phi_face[f] = phi[P]
            continue

        # Distances from cell centres to face centre
        dP = np.linalg.norm(face_centroids[f] - cell_centroids[P])

        dN = np.linalg.norm(face_centroids[f] - cell_centroids[N])

        # Numerical safety
        if dP + dN < 1.0e-15:
            wP = 0.5
            wN = 0.5
        else:
            wP = dN / (dP + dN)
            wN = dP / (dP + dN)

        phi_face[f] = wP * phi[P] + wN * phi[N]

    return phi_face

In [231]:
pressure_face_weighted = interpolate_to_faces_weighted(
    pressure,
    owner,
    neighbour,
    cell_centroids,
    face_centroids
)

In [232]:
velocity_face_weighted = interpolate_to_faces_weighted(
    velocity,
    owner,
    neighbour,
    cell_centroids,
    face_centroids
)

In [233]:
difference = np.linalg.norm(
    velocity_face_weighted - velocity_face,
    axis=1
)

print("Maximum difference :", np.max(difference))
print("Mean difference    :", np.mean(difference))

Maximum difference : 2.220446049250313e-16
Mean difference    : 1.3447179007232765e-17


In [234]:
def compute_velocity_gradient(
        velocity,
        owner,
        neighbour,
        face_area_vectors,
        cell_volumes,
        cell_centroids,
        face_centroids):
    """
    Compute the velocity gradient tensor
    using Green-Gauss reconstruction.

    Returns
    -------
    grad_u : ndarray
        Shape = (n_cells,3,3)

        grad_u[cell,i,j]

        i = velocity component
        j = spatial direction
    """

    n_cells = len(cell_volumes)

    grad_u = np.zeros((n_cells,3,3))

    # ---------- u component ----------

    u = velocity[:,0]

    u_face = interpolate_to_faces_weighted(
        u,
        owner,
        neighbour,
        cell_centroids,
        face_centroids
    )

    grad_u[:,0,:] = compute_gradient(
        u_face,
        owner,
        neighbour,
        face_area_vectors,
        cell_volumes
    )

    # ---------- v component ----------

    v = velocity[:,1]

    v_face = interpolate_to_faces_weighted(
        v,
        owner,
        neighbour,
        cell_centroids,
        face_centroids
    )

    grad_u[:,1,:] = compute_gradient(
        v_face,
        owner,
        neighbour,
        face_area_vectors,
        cell_volumes
    )

    # ---------- w component ----------

    w = velocity[:,2]

    w_face = interpolate_to_faces_weighted(
        w,
        owner,
        neighbour,
        cell_centroids,
        face_centroids
    )

    grad_u[:,2,:] = compute_gradient(
        w_face,
        owner,
        neighbour,
        face_area_vectors,
        cell_volumes
    )

    return grad_u

In [235]:
grad_u = compute_velocity_gradient(
    velocity,
    owner,
    neighbour,
    face_area_vectors,
    cell_volumes,
    cell_centroids,
    face_centroids
)

In [236]:
print(grad_u.shape)

(44172, 3, 3)


In [237]:
print(grad_u[0])

[[-5.70936019e-15  1.14187204e-14  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]]


In [238]:
def compute_strain_rate_tensor(grad_u):
    """
    Compute the strain-rate tensor.

    Parameters
    ----------
    grad_u : ndarray
        Shape = (n_cells,3,3)

    Returns
    -------
    S : ndarray
        Shape = (n_cells,3,3)
    """

    n_cells = grad_u.shape[0]

    S = np.zeros_like(grad_u)

    for cell in range(n_cells):

        S[cell] = 0.5 * (
            grad_u[cell] +
            grad_u[cell].T
        )

    return S

In [239]:
S = compute_strain_rate_tensor(grad_u)

In [240]:
print(S.shape)

print()

print(S[0])

(44172, 3, 3)

[[-5.70936019e-15  5.70936019e-15  0.00000000e+00]
 [ 5.70936019e-15  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]]


In [241]:
def compute_strain_rate_magnitude(S):
    """
    Compute |S| = sqrt(2*Sij*Sij)

    Parameters
    ----------
    S : ndarray
        Shape = (n_cells,3,3)

    Returns
    -------
    S_mag : ndarray
        Shape = (n_cells,)
    """

    # Double contraction Sij*Sij
    S_squared = np.sum(S * S, axis=(1, 2))

    S_mag = np.sqrt(2.0 * S_squared)

    return S_mag

In [242]:
S_mag = compute_strain_rate_magnitude(S)

In [243]:
print("Shape:", S_mag.shape)

print()

print("First 10 values")
print(S_mag[:10])

print()

print("Maximum:", np.max(S_mag))
print("Minimum:", np.min(S_mag))
print("Mean:", np.mean(S_mag))

Shape: (44172,)

First 10 values
[1.39850192e-14 1.16301189e-14 1.02907999e-15 6.62666408e-15
 2.18635822e-14 2.76511656e-15 5.07114091e-15 7.04881668e-16
 1.60957531e-14 7.46280307e-15]

Maximum: 1.1718148829338449e-13
Minimum: 0.0
Mean: 1.0678939482542374e-14


In [244]:
def compute_filter_width(cell_volumes):
    """
    Compute LES filter width.

    Δ = V^(1/3)

    Parameters
    ----------
    cell_volumes : ndarray

    Returns
    -------
    delta : ndarray
    """

    delta = np.cbrt(cell_volumes)

    return delta

In [245]:
delta = compute_filter_width(cell_volumes)

print(delta.shape)

print()

print(delta[:10])

print()

print("Minimum Δ =", np.min(delta))
print("Maximum Δ =", np.max(delta))
print("Mean Δ    =", np.mean(delta))

(44172,)

[0.0336138  0.02680233 0.51525198 0.03854439 0.03686065 0.10280774
 0.02775459 0.24300647 0.03524554 0.02738889]

Minimum Δ = 0.018719919593590983
Maximum Δ = 0.8046907205240692
Mean Δ    = 0.06334015746277841


In [246]:
def compute_smagorinsky_viscosity(
        delta,
        S_mag,
        Cs=0.17):
    """
    Compute Smagorinsky eddy viscosity.

    Parameters
    ----------
    delta : ndarray

    S_mag : ndarray

    Cs : float

    Returns
    -------
    nu_t : ndarray
    """

    nu_t = (Cs * delta)**2 * S_mag

    return nu_t

In [247]:
nu_t = compute_smagorinsky_viscosity(
    delta,
    S_mag
)

In [248]:
print(nu_t.shape)

print()

print(nu_t[:10])

print()

print("Maximum =", np.max(nu_t))
print("Minimum =", np.min(nu_t))
print("Mean    =", np.mean(nu_t))

(44172,)

[4.56663318e-19 2.41449865e-19 7.89562144e-18 2.84521467e-19
 8.58509665e-19 8.44623018e-19 1.12894621e-19 1.20295595e-18
 5.77852988e-19 1.61788886e-19]

Maximum = 2.3579315122650367e-17
Minimum = 0.0
Mean    = 7.182127021612893e-19


In [249]:
points
tetra
owner
neighbour
cell_centroids
cell_volumes
face_centroids
face_area_vectors
face_normals

velocity
pressure
nu_t

array([4.56663318e-19, 2.41449865e-19, 7.89562144e-18, ...,
       8.95224371e-19, 5.96163469e-19, 6.22136462e-19])

In [250]:
%whos

Variable                            Type          Data/Info
-----------------------------------------------------------
A_diff                              lil_matrix      (0, 0)	0.00034192014119<...>1)	0.00047212208816238316
A_u                                 lil_matrix      (0, 0)	0.03832173354039<...>171)	0.038395533545119334
D                                   ndarray       90207: 90207 elems, type `float64`, 721656 bytes (704.7421875 kb)
P_outlet                            float         0.0
S                                   ndarray       44172x3x3: 397548 elems, type `float64`, 3180384 bytes (3.033050537109375 Mb)
S_mag                               ndarray       44172: 44172 elems, type `float64`, 353376 bytes (345.09375 kb)
U_inlet                             ndarray       3: 3 elems, type `float64`, 24 bytes
add_convection_to_momentum_matrix   function      <function add_convection_<...>matrix at 0x7929dd82ac00>
add_diffusion_to_momentum_matrix    function      <function add

In [251]:
# ============================
# Simulation parameters
# ============================

rho = 1.0              # Density (kg/m^3)

nu = 1.0e-3            # Molecular viscosity

dt = 1.0e-3            # Time step

t = 0.0                # Initial time

t_end = 2.0            # End time

n_steps = int(t_end/dt)

print("Number of time steps =", n_steps)

Number of time steps = 2000


In [252]:
nu_eff = nu + nu_t

print(nu_eff.shape)

print("Minimum =", np.min(nu_eff))
print("Maximum =", np.max(nu_eff))

(44172,)
Minimum = 0.001
Maximum = 0.0010000000000000237


In [253]:
pressure_face = interpolate_to_faces_weighted(
    pressure,
    owner,
    neighbour,
    cell_centroids,
    face_centroids
)

In [254]:
pressure_gradient = compute_gradient(
    pressure_face,
    owner,
    neighbour,
    face_area_vectors,
    cell_volumes
)

In [255]:
pressure_face.shape

(90207,)

In [256]:
velocity_face = interpolate_to_faces_weighted(
    velocity,
    owner,
    neighbour,
    cell_centroids,
    face_centroids
)

print(velocity_face.shape)

(90207, 3)


In [257]:
mass_flux = compute_mass_flux(
    velocity_face,
    face_area_vectors,
    rho
)

print(mass_flux.shape)

print("Minimum =", np.min(mass_flux))
print("Maximum =", np.max(mass_flux))

(90207,)
Minimum = -1.2514965121430106
Maximum = 1.390261036752646


In [258]:
pressure_face = interpolate_to_faces_weighted(
    pressure,
    owner,
    neighbour,
    cell_centroids,
    face_centroids
)

pressure_gradient = compute_gradient(
    pressure_face,
    owner,
    neighbour,
    face_area_vectors,
    cell_volumes
)

print(pressure_gradient.shape)

(44172, 3)


In [259]:
def compute_convective_term(velocity_face,
                            mass_flux,
                            owner,
                            neighbour,
                            cell_volumes):
    """
    Compute the finite-volume convection term

        div(u u)

    Parameters
    ----------
    velocity_face : (n_faces,3)

    mass_flux : (n_faces,)

    owner : (n_faces,)

    neighbour : (n_faces,)

    cell_volumes : (n_cells,)

    Returns
    -------
    convection : (n_cells,3)
    """

    n_cells = len(cell_volumes)

    convection = np.zeros((n_cells,3))

    n_faces = len(owner)

    for f in range(n_faces):

        P = owner[f]
        N = neighbour[f]

        flux = mass_flux[f] * velocity_face[f]

        # Flux leaving owner
        convection[P] += flux

        # Flux entering neighbour
        if N != -1:
            convection[N] -= flux

    # Divide by cell volume
    convection /= cell_volumes[:,None]

    return convection

In [260]:
convective_term = compute_convective_term(
    velocity_face,
    mass_flux,
    owner,
    neighbour,
    cell_volumes
)

print(convective_term.shape)

(44172, 3)


In [261]:
print("Maximum =", np.max(np.abs(convective_term)))
print("Mean     =", np.mean(np.abs(convective_term)))
print("Minimum  =", np.min(np.abs(convective_term)))

Maximum = 1.1773853809478976e-13
Mean     = 1.946456761497972e-15
Minimum  = 0.0


In [262]:
def compute_diffusion_term(velocity,
                           nu_eff,
                           owner,
                           neighbour,
                           cell_centroids,
                           face_areas,
                           cell_volumes):
    """
    Compute diffusion term

        div(nu_eff grad(U))

    Returns
    -------
    diffusion : (n_cells,3)
    """

    n_cells = len(cell_volumes)

    diffusion = np.zeros((n_cells,3))

    n_faces = len(owner)

    for f in range(n_faces):

        P = owner[f]
        N = neighbour[f]

        # Boundary faces
        if N == -1:
            continue

        d = np.linalg.norm(
            cell_centroids[N] -
            cell_centroids[P]
        )

        if d < 1e-15:
            continue

        # Face viscosity
        nu_face = 0.5*(nu_eff[P] + nu_eff[N])

        # Velocity difference
        du = velocity[N] - velocity[P]

        # Diffusive flux
        flux = nu_face * face_areas[f] * du / d

        diffusion[P] += flux
        diffusion[N] -= flux

    diffusion /= cell_volumes[:,None]

    return diffusion

In [263]:
diffusion_term = compute_diffusion_term(
    velocity,
    nu_eff,
    owner,
    neighbour,
    cell_centroids,
    face_areas,
    cell_volumes
)

print(diffusion_term.shape)

(44172, 3)


In [264]:
print("Maximum =", np.max(np.abs(diffusion_term)))
print("Mean     =", np.mean(np.abs(diffusion_term)))
print("Minimum  =", np.min(np.abs(diffusion_term)))

Maximum = 0.0
Mean     = 0.0
Minimum  = 0.0


In [265]:
def assemble_momentum_rhs(velocity,
                          pressure_gradient,
                          convective_term,
                          diffusion_term,
                          cell_volumes,
                          rho):
    """
    Assemble the RHS of the momentum equation.

    Governing equation:

        V dU/dt =
            -V Convective
            -V/rho PressureGradient
            +V Diffusion

    Returns
    -------
    rhs : ndarray (n_cells,3)
    """

    rhs = np.zeros_like(velocity)

    rhs += -convective_term * cell_volumes[:, None]

    rhs += -(pressure_gradient / rho) * cell_volumes[:, None]

    rhs += diffusion_term * cell_volumes[:, None]

    return rhs

In [266]:
momentum_rhs = assemble_momentum_rhs(
    velocity,
    pressure_gradient,
    convective_term,
    diffusion_term,
    cell_volumes,
    rho
)

print(momentum_rhs.shape)

(44172, 3)


In [267]:
print("Maximum =", np.max(np.abs(momentum_rhs)))
print("Mean     =", np.mean(np.abs(momentum_rhs)))
print("Minimum  =", np.min(np.abs(momentum_rhs)))

Maximum = 6.661338147750939e-16
Mean     = 9.283898147850366e-19
Minimum  = 0.0


# PISO_FINITE_SOLVER

In [268]:
rho = 1.0
nu = 0.001

dt = 0.001

t = 0.0

t_end = 2.0

n_steps = int(t_end/dt)

print(n_steps)

2000


In [269]:
U_inlet = np.array([1.0,0.0,0.0])

P_outlet = 0.0

wall_velocity = np.zeros(3)

In [270]:
velocity = np.zeros((n_cells,3))

pressure = np.zeros(n_cells)

velocity[:,0] = 1.0

In [271]:
# ============================================
# LES: Compute eddy viscosity
# ============================================

# Velocity gradient
grad_u = compute_velocity_gradient(
    velocity,
    owner,
    neighbour,
    face_area_vectors,
    cell_volumes,
    cell_centroids,
    face_centroids
)

# Strain-rate tensor
S = compute_strain_rate_tensor(grad_u)

# Magnitude of strain-rate
S_mag = compute_strain_rate_magnitude(S)

# Filter width
delta = compute_filter_width(cell_volumes)

# Smagorinsky eddy viscosity
nu_t = compute_smagorinsky_viscosity(
    delta,
    S_mag
)

# Effective viscosity
nu_eff = nu + nu_t

print("nu_t")
print(" Minimum :", np.min(nu_t))
print(" Maximum :", np.max(nu_t))
print(" Mean    :", np.mean(nu_t))

print("\nnu_eff")
print(" Minimum :", np.min(nu_eff))
print(" Maximum :", np.max(nu_eff))
print(" Mean    :", np.mean(nu_eff))

nu_t
 Minimum : 0.0
 Maximum : 2.3579315122650367e-17
 Mean    : 7.182127021612893e-19

nu_eff
 Minimum : 0.001
 Maximum : 0.0010000000000000237
 Mean    : 0.0010000000000000009


In [272]:
def compute_diffusion_coefficients(
    nu_eff,
    owner,
    neighbour,
    cell_centroids,
    face_areas
):
    """
    Compute face diffusion coefficients

        D_f = nu_f * A_f / d_PN

    Returns
    -------
    D : (n_faces,)
    """

    n_faces = len(owner)

    D = np.zeros(n_faces)

    for f in range(n_faces):

        P = owner[f]
        N = neighbour[f]

        # Boundary face
        if N == -1:
            continue

        dPN = np.linalg.norm(
            cell_centroids[N] -
            cell_centroids[P]
        )

        if dPN < 1e-15:
            continue

        nu_face = 0.5 * (
            nu_eff[P] +
            nu_eff[N]
        )

        D[f] = nu_face * face_areas[f] / dPN

    return D

In [273]:
D = compute_diffusion_coefficients(
    nu_eff,
    owner,
    neighbour,
    cell_centroids,
    face_areas
)

print(D.shape)

print("Minimum =", np.min(D))
print("Maximum =", np.max(D))
print("Mean    =", np.mean(D))

(90207,)
Minimum = 0.0
Maximum = 0.004224044273530478
Mean    = 0.00015291831020769822


In [274]:
from scipy.sparse import lil_matrix

In [275]:
def assemble_diffusion_matrix(
        D,
        owner,
        neighbour,
        n_cells):

    """
    Assemble diffusion matrix

    Returns
    -------
    A : sparse matrix
    """

    A = lil_matrix((n_cells, n_cells))

    n_faces = len(owner)

    for f in range(n_faces):

        P = owner[f]
        N = neighbour[f]

        # Boundary face
        if N == -1:
            continue

        coeff = D[f]

        # Owner equation
        A[P, P] += coeff
        A[P, N] -= coeff

        # Neighbour equation
        A[N, N] += coeff
        A[N, P] -= coeff

    return A

In [276]:
A_diff = assemble_diffusion_matrix(
    D,
    owner,
    neighbour,
    n_cells
)

print(A_diff.shape)
print("Non-zero entries:", A_diff.nnz)

(44172, 44172)
Non-zero entries: 217134


In [277]:
symmetry_error = (A_diff - A_diff.T).tocoo()

print("Maximum asymmetry:",
      np.max(np.abs(symmetry_error.data))
      if symmetry_error.nnz > 0 else 0.0)

Maximum asymmetry: 0.0


In [278]:
def initialize_momentum_system(
        phi_old,
        cell_volumes,
        rho,
        dt):
    """
    Initialize the momentum matrix with the transient term.

    Parameters
    ----------
    phi_old : ndarray
        Velocity component from previous time step.

    cell_volumes : ndarray

    rho : float

    dt : float

    Returns
    -------
    A : sparse matrix

    b : ndarray
    """

    n_cells = len(cell_volumes)

    A = lil_matrix((n_cells, n_cells))

    b = np.zeros(n_cells)

    for P in range(n_cells):

        a_time = rho * cell_volumes[P] / dt

        A[P, P] = a_time

        b[P] = a_time * phi_old[P]

    return A, b

In [279]:
A_u, b_u = initialize_momentum_system(
    u,
    cell_volumes,
    rho,
    dt
)

print(A_u.shape)
print(b_u.shape)
print("Diagonal entries:", A_u.diagonal()[:5])
print("RHS:", b_u[:5])

(44172, 44172)
(44172,)
Diagonal entries: [3.79798134e-02 1.92538443e-02 1.36791468e+02 5.72642216e-02
 5.00828504e-02]
RHS: [3.79798134e-02 1.92538443e-02 1.36791468e+02 5.72642216e-02
 5.00828504e-02]


In [280]:
def add_diffusion_to_momentum_matrix(
    A,
    D,
    owner,
    neighbour
):
    """
    Add diffusion contributions to an existing
    momentum matrix.

    Parameters
    ----------
    A : scipy.sparse.lil_matrix
        Momentum matrix (already contains transient term)

    D : ndarray
        Face diffusion coefficients

    owner, neighbour : ndarray

    Returns
    -------
    A : updated sparse matrix
    """

    n_faces = len(owner)

    for f in range(n_faces):

        P = owner[f]
        N = neighbour[f]

        # Boundary face
        if N == -1:
            continue

        coeff = D[f]

        # Owner equation
        A[P, P] += coeff
        A[P, N] -= coeff

        # Neighbour equation
        A[N, N] += coeff
        A[N, P] -= coeff

    return A

In [281]:
A_u = add_diffusion_to_momentum_matrix(
    A_u,
    D,
    owner,
    neighbour
)

print("Matrix size :", A_u.shape)
print("Non-zeros   :", A_u.nnz)

diag = A_u.diagonal()

print("\nDiagonal statistics")
print("-------------------")
print("Minimum :", np.min(diag))
print("Maximum :", np.max(diag))
print("Mean    :", np.mean(diag))

Matrix size : (44172, 44172)
Non-zeros   : 217134

Diagonal statistics
-------------------
Minimum : 0.006761071606096681
Maximum : 521.0676427330004
Mean    : 5.648507853531796


In [282]:
print(mass_flux.shape)
print(np.min(mass_flux))
print(np.max(mass_flux))
print(np.mean(mass_flux))

(90207,)
-1.2514965121430106
1.390261036752646
-0.00013614079723567045


In [283]:
def add_convection_to_momentum_matrix(
    A,
    mass_flux,
    owner,
    neighbour
):
    """
    Add first-order upwind convection contributions
    to the momentum matrix.
    """

    n_faces = len(owner)

    for f in range(n_faces):

        P = owner[f]
        N = neighbour[f]

        # Skip boundary faces
        if N == -1:
            continue

        F = mass_flux[f]

        F_pos = max(F, 0.0)
        F_neg = min(F, 0.0)

        # Owner equation
        A[P, P] += F_pos
        A[P, N] += F_neg

        # Neighbour equation
        A[N, P] -= F_pos
        A[N, N] -= F_neg

    return A

In [284]:
face_lookup = {}

for i, face in enumerate(unique_faces):

    face_lookup[tuple(sorted(face))] = i

print("Faces in dictionary:", len(face_lookup))

Faces in dictionary: 90207


In [286]:
boundary_triangles = []
boundary_tags = []

physical_data = gmsh_mesh.cell_data["gmsh:physical"]

for block, tags in zip(gmsh_mesh.cells, physical_data):

    if block.type != "triangle":
        continue

    for tri, tag in zip(block.data, tags):

        boundary_triangles.append(tuple(sorted(tri)))
        boundary_tags.append(tag)

print("Boundary triangles:", len(boundary_triangles))

Boundary triangles: 3726


In [287]:
boundary_tag = np.full(len(unique_faces), 5, dtype=int)

for tri, tag in zip(boundary_triangles, boundary_tags):

    f = face_lookup[tri]
    boundary_tag[f] = tag

In [288]:
print("Internal :", np.sum(boundary_tag == 5))
print("Inlet    :", np.sum(boundary_tag == 1))
print("Outlet   :", np.sum(boundary_tag == 2))
print("Walls    :", np.sum(boundary_tag == 3))
print("Object   :", np.sum(boundary_tag == 4))

Internal : 86481
Inlet    : 66
Outlet   : 66
Walls    : 526
Object   : 3068


In [289]:
# =====================================================
# Boundary identifiers
# =====================================================

INLET = 1
OUTLET = 2
WALL = 3
OBJECT = 4
INTERNAL = 5

In [290]:
boundary_faces = np.where(neighbour == -1)[0]

print("Number of boundary faces:", len(boundary_faces))

print("\nFirst 10 boundary faces")

for f in boundary_faces[:10]:

    print(
        "Face =", f,
        "Owner =", owner[f],
        "Tag =", boundary_tag[f]
    )
    

Number of boundary faces: 3726

First 10 boundary faces
Face = 7 Owner = 1 Tag = 4
Face = 51 Owner = 12 Tag = 4
Face = 55 Owner = 13 Tag = 4
Face = 122 Owner = 30 Tag = 4
Face = 127 Owner = 32 Tag = 3
Face = 169 Owner = 42 Tag = 4
Face = 210 Owner = 52 Tag = 4
Face = 322 Owner = 80 Tag = 4
Face = 332 Owner = 83 Tag = 3
Face = 366 Owner = 91 Tag = 3
